In [1]:
import os
import shutil
from itertools import product
os.chdir("/home/alexandre/mestrado/graph-embedding/")

In [2]:
def get_target_dataset_names(path):
    """
    Lists all subdirectories (folders) within a given directory.

    Args:
        path (str): The path to the directory to inspect.

    Returns:
        list: A list of strings, where each string is the name of a subdirectory.
    """
    folders = []
    # Get all entries (files and directories) in the specified path
    for entry in os.listdir(path):
        # Construct the full path of the entry
        full_path = os.path.join(path, entry)
        # Check if the entry is a directory
        if os.path.isdir(full_path):
            folders.append(entry)
    return folders

In [12]:
targets = get_target_dataset_names("./preprocessed")
no_recursion_datasets = ["cora", "imdb", "uwcse"]
targets = ['yeast', 'uwcse', 'nell_finances', 'imdb', 'twitter', "cora", 'nell_sports']

custom_configs = {
    "cora": {"gammas":[0, 0.1, 0.2, 0.5, 1, 2, 2.5, 3, 3.5, 4], "lambdas": [0, 0.0001, 0.0002, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1, 2, 3]},
    "nell_finances": {"gammas":[0, 0.3, 0.5, 0.7, 1, 1.5, 2, 2.5, 3, 3.5, 4], "lambdas": [0, 0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1, 0.15, 0.2, 0.3, 0.5, 1]},
    "uwcse": {"gammas":[0, 0.5, 1, 1.5, 2, 2.5, 3, 4], "lambdas": [0, 0.01, 0.02, 0.03, 0.04, 0.05, 0.1, 0.2, 0.5, 1]},
    "yeast": {"gammas":[0, 0.05, 0.1, 0,2, 0.5, 0.7, 1, 1.5, 2, 3, 4], "lambdas": [0, 0.01, 0.02, 0.03, 0.05, 0.07, 0.1, 0.2, 0.5, 1]}
}

configs = {}

for target in custom_configs:
    configs[target] = [
        {"focal_loss_gamma": g, "boxcox_lambda": l}
        for g, l in product(custom_configs[target]["gammas"], custom_configs[target]["lambdas"])
    ]

In [4]:
configs

{'cora': [{'focal_loss_gamma': 0.2, 'boxcox_lambda': 0},
  {'focal_loss_gamma': 0.2, 'boxcox_lambda': 0.0001},
  {'focal_loss_gamma': 0.2, 'boxcox_lambda': 0.0002},
  {'focal_loss_gamma': 0.2, 'boxcox_lambda': 0.005},
  {'focal_loss_gamma': 0.2, 'boxcox_lambda': 0.01},
  {'focal_loss_gamma': 0.2, 'boxcox_lambda': 0.02},
  {'focal_loss_gamma': 0.2, 'boxcox_lambda': 0.05},
  {'focal_loss_gamma': 0.2, 'boxcox_lambda': 0.1},
  {'focal_loss_gamma': 0.2, 'boxcox_lambda': 0.2},
  {'focal_loss_gamma': 0.2, 'boxcox_lambda': 0.5},
  {'focal_loss_gamma': 0.5, 'boxcox_lambda': 0},
  {'focal_loss_gamma': 0.5, 'boxcox_lambda': 0.0001},
  {'focal_loss_gamma': 0.5, 'boxcox_lambda': 0.0002},
  {'focal_loss_gamma': 0.5, 'boxcox_lambda': 0.005},
  {'focal_loss_gamma': 0.5, 'boxcox_lambda': 0.01},
  {'focal_loss_gamma': 0.5, 'boxcox_lambda': 0.02},
  {'focal_loss_gamma': 0.5, 'boxcox_lambda': 0.05},
  {'focal_loss_gamma': 0.5, 'boxcox_lambda': 0.1},
  {'focal_loss_gamma': 0.5, 'boxcox_lambda': 0.2},
  {'f

In [9]:
from itertools import product

# Define parameter ranges
gammas = [0, 0.5, 1, 2, 3, 4]
lambdas = [0, 0.02, 0.05, 0.1, 0.2, 0.5, 1]

# Generate all combinations
configs = [
    {"focal_loss_gamma": g, "boxcox_lambda": l}
    for g, l in product(gammas, lambdas)
]

# Example: print the total number and a few samples
print(f"Total configurations: {len(configs)}")
print(configs[:5])  # show first 5

Total configurations: 42
[{'focal_loss_gamma': 0, 'boxcox_lambda': 0}, {'focal_loss_gamma': 0, 'boxcox_lambda': 0.02}, {'focal_loss_gamma': 0, 'boxcox_lambda': 0.05}, {'focal_loss_gamma': 0, 'boxcox_lambda': 0.1}, {'focal_loss_gamma': 0, 'boxcox_lambda': 0.2}]


## No Transfer Learning - Small Dataset - Original RDN Boost

In [11]:
from models.main import default_workflow
from models.utils import update_experiment_state

base_path = "/home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning"

for target in targets:
    out = default_workflow(base_path, target, {"rdn_version": "rdn"}, base_path, use_recursion=target not in no_recursion_datasets)
    update_experiment_state(os.path.join(base_path, f"small_dataset_results_original_{target}.json"), {"rdn_version": "rdn", "results": out})    

java  -jar /home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/srlearn/BoostSRL.jar -l -train /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train -target teamplayssport -trees 10 -negPosRatio 2 -focalLossAlpha 1  -focalLossGamma 2  -boxCoxLambda 0  -maxTreeDepthInLiterals -1  -rdnVersion rdn  2>&1 | tee /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train_output.txt


/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no classes that implement `__sklearn_tags__` in the MRO and/or all classes in the MRO call `super().__sklearn_tags__()`. Make sure to inherit from `BaseEstimator` which implements `__sklearn_tags__` (or alternatively define `__sklearn_tags__` but we don't recommend this approach). Note that `BaseEstimator` needs to be on the right side of other Mixins in the inheritance order. The default are now used instead since retrieving tags failed. This warning will be replaced by an error in 1.8.
  warnings.warn(
/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no cla

{'f1': 0.982987, 'recall': 0.984848, 'precision': 0.981132, 'cll': -0.1297, 'auc_pr': 0.992413, 'auc_roc': 0.996829}
java  -jar /home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/srlearn/BoostSRL.jar -l -train /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train -target teamplayssport -trees 10 -negPosRatio 2 -focalLossAlpha 1  -focalLossGamma 2  -boxCoxLambda 0  -maxTreeDepthInLiterals -1  -rdnVersion rdn  2>&1 | tee /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train_output.txt


/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no classes that implement `__sklearn_tags__` in the MRO and/or all classes in the MRO call `super().__sklearn_tags__()`. Make sure to inherit from `BaseEstimator` which implements `__sklearn_tags__` (or alternatively define `__sklearn_tags__` but we don't recommend this approach). Note that `BaseEstimator` needs to be on the right side of other Mixins in the inheritance order. The default are now used instead since retrieving tags failed. This warning will be replaced by an error in 1.8.
  warnings.warn(
/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no cla

{'f1': 0.981132, 'recall': 0.984848, 'precision': 0.977444, 'cll': -0.131158, 'auc_pr': 0.994508, 'auc_roc': 0.99661}
java  -jar /home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/srlearn/BoostSRL.jar -l -train /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train -target teamplayssport -trees 10 -negPosRatio 2 -focalLossAlpha 1  -focalLossGamma 2  -boxCoxLambda 0  -maxTreeDepthInLiterals -1  -rdnVersion rdn  2>&1 | tee /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train_output.txt


/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no classes that implement `__sklearn_tags__` in the MRO and/or all classes in the MRO call `super().__sklearn_tags__()`. Make sure to inherit from `BaseEstimator` which implements `__sklearn_tags__` (or alternatively define `__sklearn_tags__` but we don't recommend this approach). Note that `BaseEstimator` needs to be on the right side of other Mixins in the inheritance order. The default are now used instead since retrieving tags failed. This warning will be replaced by an error in 1.8.
  warnings.warn(
/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no cla

... (153 outputs left out)

/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no classes that implement `__sklearn_tags__` in the MRO and/or all classes in the MRO call `super().__sklearn_tags__()`. Make sure to inherit from `BaseEstimator` which implements `__sklearn_tags__` (or alternatively define `__sklearn_tags__` but we don't recommend this approach). Note that `BaseEstimator` needs to be on the right side of other Mixins in the inheritance order. The default are now used instead since retrieving tags failed. This warning will be replaced by an error in 1.8.
  warnings.warn(
/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no cla

{'inference_time_ms': 394, 'f1': 0.850877, 'recall': 1.0, 'precision': 0.740458, 'cll': -0.301725, 'auc_pr': 0.826774, 'auc_roc': 0.944574}
java  -jar /home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/srlearn/BoostSRL.jar -l -train /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train -target advisedby -trees 10 -negPosRatio 2 -focalLossAlpha 1  -focalLossGamma 2  -boxCoxLambda 0  -maxTreeDepthInLiterals -1  -rdnVersion rdn  2>&1 | tee /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train_output.txt


/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no classes that implement `__sklearn_tags__` in the MRO and/or all classes in the MRO call `super().__sklearn_tags__()`. Make sure to inherit from `BaseEstimator` which implements `__sklearn_tags__` (or alternatively define `__sklearn_tags__` but we don't recommend this approach). Note that `BaseEstimator` needs to be on the right side of other Mixins in the inheritance order. The default are now used instead since retrieving tags failed. This warning will be replaced by an error in 1.8.
  warnings.warn(
/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no cla

{'inference_time_ms': 387, 'f1': 0.850877, 'recall': 1.0, 'precision': 0.740458, 'cll': -0.308535, 'auc_pr': 0.826945, 'auc_roc': 0.943591}


In [6]:
from models.main import default_workflow
from models.utils import update_experiment_state

base_path = "/home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning"

for target in targets:
    out = default_workflow(base_path, target, {"rdn_version": "rdn"}, base_path, use_recursion=True)
    update_experiment_state(os.path.join(base_path, f"small_dataset_results_original_{target}_recursion.json"), {"rdn_version": "rdn", "results": out})    

java  -jar /home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/srlearn/BoostSRL.jar -l -train /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train -target teamplayssport -trees 10 -negPosRatio 2 -focalLossAlpha 0.66  -focalLossGamma 2  -boxCoxLambda 0  -maxTreeDepthInLiterals -1  -rdnVersion rdn  2>&1 | tee /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train_output.txt


/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no classes that implement `__sklearn_tags__` in the MRO and/or all classes in the MRO call `super().__sklearn_tags__()`. Make sure to inherit from `BaseEstimator` which implements `__sklearn_tags__` (or alternatively define `__sklearn_tags__` but we don't recommend this approach). Note that `BaseEstimator` needs to be on the right side of other Mixins in the inheritance order. The default are now used instead since retrieving tags failed. This warning will be replaced by an error in 1.8.
  warnings.warn(
/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no cla

java  -jar /home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/srlearn/BoostSRL.jar -l -train /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train -target teamplayssport -trees 10 -negPosRatio 2 -focalLossAlpha 0.66  -focalLossGamma 2  -boxCoxLambda 0  -maxTreeDepthInLiterals -1  -rdnVersion rdn  2>&1 | tee /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train_output.txt


/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no classes that implement `__sklearn_tags__` in the MRO and/or all classes in the MRO call `super().__sklearn_tags__()`. Make sure to inherit from `BaseEstimator` which implements `__sklearn_tags__` (or alternatively define `__sklearn_tags__` but we don't recommend this approach). Note that `BaseEstimator` needs to be on the right side of other Mixins in the inheritance order. The default are now used instead since retrieving tags failed. This warning will be replaced by an error in 1.8.
  warnings.warn(
/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no cla

java  -jar /home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/srlearn/BoostSRL.jar -l -train /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train -target teamplayssport -trees 10 -negPosRatio 2 -focalLossAlpha 0.66  -focalLossGamma 2  -boxCoxLambda 0  -maxTreeDepthInLiterals -1  -rdnVersion rdn  2>&1 | tee /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train_output.txt


/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no classes that implement `__sklearn_tags__` in the MRO and/or all classes in the MRO call `super().__sklearn_tags__()`. Make sure to inherit from `BaseEstimator` which implements `__sklearn_tags__` (or alternatively define `__sklearn_tags__` but we don't recommend this approach). Note that `BaseEstimator` needs to be on the right side of other Mixins in the inheritance order. The default are now used instead since retrieving tags failed. This warning will be replaced by an error in 1.8.
  warnings.warn(
/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no cla

... (152 outputs left out)

java  -jar /home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/srlearn/BoostSRL.jar -l -train /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train -target advisedby -trees 10 -negPosRatio 2 -focalLossAlpha 0.66  -focalLossGamma 2  -boxCoxLambda 0  -maxTreeDepthInLiterals -1  -rdnVersion rdn  2>&1 | tee /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train_output.txt


/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no classes that implement `__sklearn_tags__` in the MRO and/or all classes in the MRO call `super().__sklearn_tags__()`. Make sure to inherit from `BaseEstimator` which implements `__sklearn_tags__` (or alternatively define `__sklearn_tags__` but we don't recommend this approach). Note that `BaseEstimator` needs to be on the right side of other Mixins in the inheritance order. The default are now used instead since retrieving tags failed. This warning will be replaced by an error in 1.8.
  warnings.warn(
/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no cla

java  -jar /home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/srlearn/BoostSRL.jar -l -train /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train -target advisedby -trees 10 -negPosRatio 2 -focalLossAlpha 0.66  -focalLossGamma 2  -boxCoxLambda 0  -maxTreeDepthInLiterals -1  -rdnVersion rdn  2>&1 | tee /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train_output.txt


/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no classes that implement `__sklearn_tags__` in the MRO and/or all classes in the MRO call `super().__sklearn_tags__()`. Make sure to inherit from `BaseEstimator` which implements `__sklearn_tags__` (or alternatively define `__sklearn_tags__` but we don't recommend this approach). Note that `BaseEstimator` needs to be on the right side of other Mixins in the inheritance order. The default are now used instead since retrieving tags failed. This warning will be replaced by an error in 1.8.
  warnings.warn(
/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no cla

## No Transfer Learning - Large Dataset - Original RDN Boost

In [12]:
from models.main import default_workflow
from models.utils import update_experiment_state

base_path = "/home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning"

for target in targets:
    out = default_workflow(base_path, target, {"rdn_version": "rdn"}, base_path, small_dataset = False, use_recursion=target not in no_recursion_datasets)
    update_experiment_state(os.path.join(base_path, f"large_dataset_results_original_{target}.json"), {"rdn_version": "rdn", "results": out})    

java  -jar /home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/srlearn/BoostSRL.jar -l -train /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train -target teamplayssport -trees 10 -negPosRatio 2 -focalLossAlpha 1  -focalLossGamma 2  -boxCoxLambda 0  -maxTreeDepthInLiterals -1  -rdnVersion rdn  2>&1 | tee /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train_output.txt


/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no classes that implement `__sklearn_tags__` in the MRO and/or all classes in the MRO call `super().__sklearn_tags__()`. Make sure to inherit from `BaseEstimator` which implements `__sklearn_tags__` (or alternatively define `__sklearn_tags__` but we don't recommend this approach). Note that `BaseEstimator` needs to be on the right side of other Mixins in the inheritance order. The default are now used instead since retrieving tags failed. This warning will be replaced by an error in 1.8.
  warnings.warn(
/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no cla

{'f1': 0.966292, 'recall': 0.977273, 'precision': 0.955556, 'cll': -0.147718, 'auc_pr': 0.989533, 'auc_roc': 0.995279}
java  -jar /home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/srlearn/BoostSRL.jar -l -train /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train -target teamplayssport -trees 10 -negPosRatio 2 -focalLossAlpha 1  -focalLossGamma 2  -boxCoxLambda 0  -maxTreeDepthInLiterals -1  -rdnVersion rdn  2>&1 | tee /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train_output.txt


/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no classes that implement `__sklearn_tags__` in the MRO and/or all classes in the MRO call `super().__sklearn_tags__()`. Make sure to inherit from `BaseEstimator` which implements `__sklearn_tags__` (or alternatively define `__sklearn_tags__` but we don't recommend this approach). Note that `BaseEstimator` needs to be on the right side of other Mixins in the inheritance order. The default are now used instead since retrieving tags failed. This warning will be replaced by an error in 1.8.
  warnings.warn(
/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no cla

{'f1': 0.963235, 'recall': 0.992424, 'precision': 0.935714, 'cll': -0.151088, 'auc_pr': 0.987364, 'auc_roc': 0.989239}
java  -jar /home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/srlearn/BoostSRL.jar -l -train /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train -target teamplayssport -trees 10 -negPosRatio 2 -focalLossAlpha 1  -focalLossGamma 2  -boxCoxLambda 0  -maxTreeDepthInLiterals -1  -rdnVersion rdn  2>&1 | tee /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train_output.txt


/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no classes that implement `__sklearn_tags__` in the MRO and/or all classes in the MRO call `super().__sklearn_tags__()`. Make sure to inherit from `BaseEstimator` which implements `__sklearn_tags__` (or alternatively define `__sklearn_tags__` but we don't recommend this approach). Note that `BaseEstimator` needs to be on the right side of other Mixins in the inheritance order. The default are now used instead since retrieving tags failed. This warning will be replaced by an error in 1.8.
  warnings.warn(
/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no cla

... (153 outputs left out)

/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no classes that implement `__sklearn_tags__` in the MRO and/or all classes in the MRO call `super().__sklearn_tags__()`. Make sure to inherit from `BaseEstimator` which implements `__sklearn_tags__` (or alternatively define `__sklearn_tags__` but we don't recommend this approach). Note that `BaseEstimator` needs to be on the right side of other Mixins in the inheritance order. The default are now used instead since retrieving tags failed. This warning will be replaced by an error in 1.8.
  warnings.warn(
/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no cla

{'inference_time_ms': 329, 'f1': 0.842105, 'recall': 1.0, 'precision': 0.727273, 'cll': -0.310301, 'auc_pr': 0.861805, 'auc_roc': 0.932617}
java  -jar /home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/srlearn/BoostSRL.jar -l -train /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train -target advisedby -trees 10 -negPosRatio 2 -focalLossAlpha 1  -focalLossGamma 2  -boxCoxLambda 0  -maxTreeDepthInLiterals -1  -rdnVersion rdn  2>&1 | tee /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train_output.txt


/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no classes that implement `__sklearn_tags__` in the MRO and/or all classes in the MRO call `super().__sklearn_tags__()`. Make sure to inherit from `BaseEstimator` which implements `__sklearn_tags__` (or alternatively define `__sklearn_tags__` but we don't recommend this approach). Note that `BaseEstimator` needs to be on the right side of other Mixins in the inheritance order. The default are now used instead since retrieving tags failed. This warning will be replaced by an error in 1.8.
  warnings.warn(
/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no cla

{'inference_time_ms': 304, 'f1': 0.842105, 'recall': 1.0, 'precision': 0.727273, 'cll': -0.310679, 'auc_pr': 0.841928, 'auc_roc': 0.927734}


In [ ]:
from models.main import default_workflow
from models.utils import update_experiment_state

base_path = "/home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning"

for target in targets:
    out = default_workflow(base_path, target, {"rdn_version": "rdn"}, base_path, small_dataset = False, use_recursion=True)
    update_experiment_state(os.path.join(base_path, f"large_dataset_results_original_{target}_recursion.json"), {"rdn_version": "rdn", "results": out})    

java  -jar /home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/srlearn/BoostSRL.jar -l -train /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train -target teamplayssport -trees 10 -negPosRatio 2 -focalLossAlpha 0.66  -focalLossGamma 2  -boxCoxLambda 0  -maxTreeDepthInLiterals -1  -rdnVersion rdn  2>&1 | tee /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train_output.txt


/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no classes that implement `__sklearn_tags__` in the MRO and/or all classes in the MRO call `super().__sklearn_tags__()`. Make sure to inherit from `BaseEstimator` which implements `__sklearn_tags__` (or alternatively define `__sklearn_tags__` but we don't recommend this approach). Note that `BaseEstimator` needs to be on the right side of other Mixins in the inheritance order. The default are now used instead since retrieving tags failed. This warning will be replaced by an error in 1.8.
  warnings.warn(
/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no cla

java  -jar /home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/srlearn/BoostSRL.jar -l -train /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train -target teamplayssport -trees 10 -negPosRatio 2 -focalLossAlpha 0.66  -focalLossGamma 2  -boxCoxLambda 0  -maxTreeDepthInLiterals -1  -rdnVersion rdn  2>&1 | tee /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train_output.txt


/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no classes that implement `__sklearn_tags__` in the MRO and/or all classes in the MRO call `super().__sklearn_tags__()`. Make sure to inherit from `BaseEstimator` which implements `__sklearn_tags__` (or alternatively define `__sklearn_tags__` but we don't recommend this approach). Note that `BaseEstimator` needs to be on the right side of other Mixins in the inheritance order. The default are now used instead since retrieving tags failed. This warning will be replaced by an error in 1.8.
  warnings.warn(
/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no cla

java  -jar /home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/srlearn/BoostSRL.jar -l -train /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train -target teamplayssport -trees 10 -negPosRatio 2 -focalLossAlpha 0.66  -focalLossGamma 2  -boxCoxLambda 0  -maxTreeDepthInLiterals -1  -rdnVersion rdn  2>&1 | tee /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train_output.txt


/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no classes that implement `__sklearn_tags__` in the MRO and/or all classes in the MRO call `super().__sklearn_tags__()`. Make sure to inherit from `BaseEstimator` which implements `__sklearn_tags__` (or alternatively define `__sklearn_tags__` but we don't recommend this approach). Note that `BaseEstimator` needs to be on the right side of other Mixins in the inheritance order. The default are now used instead since retrieving tags failed. This warning will be replaced by an error in 1.8.
  warnings.warn(
/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no cla

... (51 outputs left out)

/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no classes that implement `__sklearn_tags__` in the MRO and/or all classes in the MRO call `super().__sklearn_tags__()`. Make sure to inherit from `BaseEstimator` which implements `__sklearn_tags__` (or alternatively define `__sklearn_tags__` but we don't recommend this approach). Note that `BaseEstimator` needs to be on the right side of other Mixins in the inheritance order. The default are now used instead since retrieving tags failed. This warning will be replaced by an error in 1.8.
  warnings.warn(
/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no cla

java  -jar /home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/srlearn/BoostSRL.jar -l -train /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train -target samevenue -trees 10 -negPosRatio 2 -focalLossAlpha 0.66  -focalLossGamma 2  -boxCoxLambda 0  -maxTreeDepthInLiterals -1  -rdnVersion rdn  2>&1 | tee /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train_output.txt


/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no classes that implement `__sklearn_tags__` in the MRO and/or all classes in the MRO call `super().__sklearn_tags__()`. Make sure to inherit from `BaseEstimator` which implements `__sklearn_tags__` (or alternatively define `__sklearn_tags__` but we don't recommend this approach). Note that `BaseEstimator` needs to be on the right side of other Mixins in the inheritance order. The default are now used instead since retrieving tags failed. This warning will be replaced by an error in 1.8.
  warnings.warn(
/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no cla

java  -jar /home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/srlearn/BoostSRL.jar -l -train /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train -target samevenue -trees 10 -negPosRatio 2 -focalLossAlpha 0.66  -focalLossGamma 2  -boxCoxLambda 0  -maxTreeDepthInLiterals -1  -rdnVersion rdn  2>&1 | tee /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train_output.txt


## No Transfer Learning - Small Dataset - GFL RDN Boost

In [3]:
def skip_check(experiments_map_path, config):
    import json
    
    if os.path.exists(experiments_map_path):
        with open(experiments_map_path, "r") as f:
            results = json.load(f)
            
            for result in results:
                if all([result[setting] == config[setting] for setting in config]):
                    return True
                    
    return False
                    

In [7]:
base_path = "/home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning"
for config in configs:
    print(skip_check(os.path.join(base_path, f"small_dataset_results_gfl_cora.json"), config))

True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False


In [14]:
from models.main import default_workflow
from models.utils import update_experiment_state

base_path = "/home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning"

for target in targets:
    for config in configs.get(target, []):
        experiments_file_path = os.path.join(base_path, f"small_dataset_results_gfl_{target}.json")

        if skip_check(experiments_file_path, config):
            continue
            
        print(config)
        out = default_workflow(base_path, target, {**config, "rdn_version": "gflRdn"}, base_path, use_recursion=target not in no_recursion_datasets)
        update_experiment_state(experiments_file_path, {**config, "rdn_version": "gflRdn", "results": out})
        

{'focal_loss_gamma': 0, 'boxcox_lambda': 0.0001}
java  -jar /home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/srlearn/BoostSRL.jar -l -train /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train -target samevenue -trees 10 -negPosRatio 2 -focalLossAlpha 1  -focalLossGamma 0  -boxCoxLambda 0.0001  -maxTreeDepthInLiterals -1  -rdnVersion gflRdn  2>&1 | tee /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train_output.txt


/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no classes that implement `__sklearn_tags__` in the MRO and/or all classes in the MRO call `super().__sklearn_tags__()`. Make sure to inherit from `BaseEstimator` which implements `__sklearn_tags__` (or alternatively define `__sklearn_tags__` but we don't recommend this approach). Note that `BaseEstimator` needs to be on the right side of other Mixins in the inheritance order. The default are now used instead since retrieving tags failed. This warning will be replaced by an error in 1.8.
  warnings.warn(
/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no cla

{'f1': 0.633757, 'recall': 0.959607, 'precision': 0.473106, 'cll': -0.691938, 'auc_pr': 0.49188, 'auc_roc': 0.553202}
java  -jar /home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/srlearn/BoostSRL.jar -l -train /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train -target samevenue -trees 10 -negPosRatio 2 -focalLossAlpha 1  -focalLossGamma 0  -boxCoxLambda 0.0001  -maxTreeDepthInLiterals -1  -rdnVersion gflRdn  2>&1 | tee /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train_output.txt


/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no classes that implement `__sklearn_tags__` in the MRO and/or all classes in the MRO call `super().__sklearn_tags__()`. Make sure to inherit from `BaseEstimator` which implements `__sklearn_tags__` (or alternatively define `__sklearn_tags__` but we don't recommend this approach). Note that `BaseEstimator` needs to be on the right side of other Mixins in the inheritance order. The default are now used instead since retrieving tags failed. This warning will be replaced by an error in 1.8.
  warnings.warn(
/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no cla

{'f1': 0.633757, 'recall': 0.959607, 'precision': 0.473106, 'cll': -0.693254, 'auc_pr': 0.492301, 'auc_roc': 0.554081}
java  -jar /home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/srlearn/BoostSRL.jar -l -train /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train -target samevenue -trees 10 -negPosRatio 2 -focalLossAlpha 1  -focalLossGamma 0  -boxCoxLambda 0.0001  -maxTreeDepthInLiterals -1  -rdnVersion gflRdn  2>&1 | tee /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train_output.txt


/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no classes that implement `__sklearn_tags__` in the MRO and/or all classes in the MRO call `super().__sklearn_tags__()`. Make sure to inherit from `BaseEstimator` which implements `__sklearn_tags__` (or alternatively define `__sklearn_tags__` but we don't recommend this approach). Note that `BaseEstimator` needs to be on the right side of other Mixins in the inheritance order. The default are now used instead since retrieving tags failed. This warning will be replaced by an error in 1.8.
  warnings.warn(
/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no cla

... (351 outputs left out)

/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no classes that implement `__sklearn_tags__` in the MRO and/or all classes in the MRO call `super().__sklearn_tags__()`. Make sure to inherit from `BaseEstimator` which implements `__sklearn_tags__` (or alternatively define `__sklearn_tags__` but we don't recommend this approach). Note that `BaseEstimator` needs to be on the right side of other Mixins in the inheritance order. The default are now used instead since retrieving tags failed. This warning will be replaced by an error in 1.8.
  warnings.warn(
/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no cla

{'f1': 0.600153, 'recall': 0.952169, 'precision': 0.438165, 'cll': -0.672748, 'auc_pr': 0.458856, 'auc_roc': 0.545121}
java  -jar /home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/srlearn/BoostSRL.jar -l -train /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train -target samevenue -trees 10 -negPosRatio 2 -focalLossAlpha 1  -focalLossGamma 3.5  -boxCoxLambda 3  -maxTreeDepthInLiterals -1  -rdnVersion gflRdn  2>&1 | tee /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train_output.txt


/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no classes that implement `__sklearn_tags__` in the MRO and/or all classes in the MRO call `super().__sklearn_tags__()`. Make sure to inherit from `BaseEstimator` which implements `__sklearn_tags__` (or alternatively define `__sklearn_tags__` but we don't recommend this approach). Note that `BaseEstimator` needs to be on the right side of other Mixins in the inheritance order. The default are now used instead since retrieving tags failed. This warning will be replaced by an error in 1.8.
  warnings.warn(
/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no cla

{'f1': 0.600153, 'recall': 0.952169, 'precision': 0.438165, 'cll': -0.672702, 'auc_pr': 0.461378, 'auc_roc': 0.54794}


In [ ]:
from models.main import default_workflow
from models.utils import update_experiment_state

base_path = "/home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning"

for target in ["cora"]:
    for config in configs:
        print(config)
        out = default_workflow(base_path, target, {**config, "rdn_version": "gflRdn"}, base_path, use_recursion=False)
        update_experiment_state(os.path.join(base_path, f"small_dataset_results_gfl_{target}.json"), {**config, "rdn_version": "gflRdn", "results": out})
    

{'focal_loss_gamma': 0, 'boxcox_lambda': 0}
java  -jar /home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/srlearn/BoostSRL.jar -l -train /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train -target samevenue -trees 10 -negPosRatio 2 -focalLossAlpha 0.66  -focalLossGamma 0  -boxCoxLambda 0  -maxTreeDepthInLiterals -1  -rdnVersion gflRdn  2>&1 | tee /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train_output.txt


/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no classes that implement `__sklearn_tags__` in the MRO and/or all classes in the MRO call `super().__sklearn_tags__()`. Make sure to inherit from `BaseEstimator` which implements `__sklearn_tags__` (or alternatively define `__sklearn_tags__` but we don't recommend this approach). Note that `BaseEstimator` needs to be on the right side of other Mixins in the inheritance order. The default are now used instead since retrieving tags failed. This warning will be replaced by an error in 1.8.
  warnings.warn(
/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no cla

{'f1': 0.633757, 'recall': 0.959607, 'precision': 0.473106, 'cll': -0.674509, 'auc_pr': 0.492446, 'auc_roc': 0.554079}
java  -jar /home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/srlearn/BoostSRL.jar -l -train /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train -target samevenue -trees 10 -negPosRatio 2 -focalLossAlpha 0.66  -focalLossGamma 0  -boxCoxLambda 0  -maxTreeDepthInLiterals -1  -rdnVersion gflRdn  2>&1 | tee /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train_output.txt


/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no classes that implement `__sklearn_tags__` in the MRO and/or all classes in the MRO call `super().__sklearn_tags__()`. Make sure to inherit from `BaseEstimator` which implements `__sklearn_tags__` (or alternatively define `__sklearn_tags__` but we don't recommend this approach). Note that `BaseEstimator` needs to be on the right side of other Mixins in the inheritance order. The default are now used instead since retrieving tags failed. This warning will be replaced by an error in 1.8.
  warnings.warn(
/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no cla

{'f1': 0.633757, 'recall': 0.959607, 'precision': 0.473106, 'cll': -0.674528, 'auc_pr': 0.492388, 'auc_roc': 0.554109}
java  -jar /home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/srlearn/BoostSRL.jar -l -train /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train -target samevenue -trees 10 -negPosRatio 2 -focalLossAlpha 0.66  -focalLossGamma 0  -boxCoxLambda 0  -maxTreeDepthInLiterals -1  -rdnVersion gflRdn  2>&1 | tee /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train_output.txt


/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no classes that implement `__sklearn_tags__` in the MRO and/or all classes in the MRO call `super().__sklearn_tags__()`. Make sure to inherit from `BaseEstimator` which implements `__sklearn_tags__` (or alternatively define `__sklearn_tags__` but we don't recommend this approach). Note that `BaseEstimator` needs to be on the right side of other Mixins in the inheritance order. The default are now used instead since retrieving tags failed. This warning will be replaced by an error in 1.8.
  warnings.warn(
/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no cla

... (47 outputs left out)

/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no classes that implement `__sklearn_tags__` in the MRO and/or all classes in the MRO call `super().__sklearn_tags__()`. Make sure to inherit from `BaseEstimator` which implements `__sklearn_tags__` (or alternatively define `__sklearn_tags__` but we don't recommend this approach). Note that `BaseEstimator` needs to be on the right side of other Mixins in the inheritance order. The default are now used instead since retrieving tags failed. This warning will be replaced by an error in 1.8.
  warnings.warn(
/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no cla

{'f1': 0.631471, 'recall': 0.945815, 'precision': 0.473951, 'cll': -0.672448, 'auc_pr': 0.507792, 'auc_roc': 0.575681}
java  -jar /home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/srlearn/BoostSRL.jar -l -train /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train -target samevenue -trees 10 -negPosRatio 2 -focalLossAlpha 0.66  -focalLossGamma 0  -boxCoxLambda 0.1  -maxTreeDepthInLiterals -1  -rdnVersion gflRdn  2>&1 | tee /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train_output.txt


/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no classes that implement `__sklearn_tags__` in the MRO and/or all classes in the MRO call `super().__sklearn_tags__()`. Make sure to inherit from `BaseEstimator` which implements `__sklearn_tags__` (or alternatively define `__sklearn_tags__` but we don't recommend this approach). Note that `BaseEstimator` needs to be on the right side of other Mixins in the inheritance order. The default are now used instead since retrieving tags failed. This warning will be replaced by an error in 1.8.
  warnings.warn(
/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no cla

{'f1': 0.600077, 'recall': 0.952169, 'precision': 0.438083, 'cll': -0.74044, 'auc_pr': 0.45984, 'auc_roc': 0.547834}
java  -jar /home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/srlearn/BoostSRL.jar -l -train /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train -target samevenue -trees 10 -negPosRatio 2 -focalLossAlpha 0.66  -focalLossGamma 0  -boxCoxLambda 0.1  -maxTreeDepthInLiterals -1  -rdnVersion gflRdn  2>&1 | tee /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train_output.txt


In [23]:
targets

['nell_sports', 'imdb', 'cora', 'twitter', 'nell_finances', 'yeast', 'uwcse']

In [16]:
for target in targets:
    print(target)

nell_sports
imdb
cora
twitter
nell_finances
yeast
uwcse


## No Transfer Learning - Large Dataset - GFL RDN Boost

In [4]:
targets = get_target_dataset_names("./preprocessed")
no_recursion_datasets = ["cora", "imdb", "uwcse"]
targets = ['yeast', 'uwcse', 'nell_finances', 'imdb', 'twitter', "cora", 'nell_sports']

custom_configs = {
    "nell_finances": {"gammas":[0, 0.5, 1, 1.5, 2, 2.5, 3, 4], "lambdas": [0, 0.02, 0.05, 0.09, 0.1, 1.15, 0.2, 0.5, 1]},
    "uwcse": {"gammas":[0, 0.01, 0.02, 0.1, 0.5, 1, 2, 3, 4], "lambdas": [0, 0.02, 0.05, 0.1, 0.2, 0.4, 0.5, 0.6, 1]},
    "yeast": {"gammas":[0, 0.5, 0.7, 1, 1.5, 2, 2.5, 3, 3.5, 4], "lambdas": [0, 0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.1, 0.2, 0.5, 1]}
}

# gammas = [0, 0.5, 1, 2, 3, 4]
# lambdas = [0, 0.02, 0.05, 0.1, 0.2, 0.5, 1]

configs = {}

for target in custom_configs:
    configs[target] = [
        {"focal_loss_gamma": g, "boxcox_lambda": l}
        for g, l in product(custom_configs[target]["gammas"], custom_configs[target]["lambdas"])
    ]

In [ ]:
from models.main import default_workflow
from models.utils import update_experiment_state

base_path = "/home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning"

for target in targets:
    for config in configs.get(target, []):
        experiments_file_path = os.path.join(base_path, f"large_dataset_results_gfl_{target}.json")
        
        if skip_check(experiments_file_path, config):
            continue
            
        print(config)
        out = default_workflow(base_path, target, {**config, "rdn_version": "gflRdn"}, base_path, small_dataset = False, use_recursion=target not in no_recursion_datasets)
        update_experiment_state(experiments_file_path, {**config, "rdn_version": "gflRdn", "results": out})
        

{'focal_loss_gamma': 4, 'boxcox_lambda': 0.09}
java  -jar /home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/srlearn/BoostSRL.jar -l -train /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train -target companyeconomicsector -trees 10 -negPosRatio 2 -focalLossAlpha 1  -focalLossGamma 4  -boxCoxLambda 0.09  -maxTreeDepthInLiterals -1  -rdnVersion gflRdn  2>&1 | tee /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train_output.txt


/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no classes that implement `__sklearn_tags__` in the MRO and/or all classes in the MRO call `super().__sklearn_tags__()`. Make sure to inherit from `BaseEstimator` which implements `__sklearn_tags__` (or alternatively define `__sklearn_tags__` but we don't recommend this approach). Note that `BaseEstimator` needs to be on the right side of other Mixins in the inheritance order. The default are now used instead since retrieving tags failed. This warning will be replaced by an error in 1.8.
  warnings.warn(
/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no cla

{'inference_time_ms': 769, 'f1': 0.563934, 'recall': 0.996139, 'precision': 0.393293, 'cll': -0.611055, 'auc_pr': 0.621776, 'auc_roc': 0.711301}
java  -jar /home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/srlearn/BoostSRL.jar -l -train /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train -target companyeconomicsector -trees 10 -negPosRatio 2 -focalLossAlpha 1  -focalLossGamma 4  -boxCoxLambda 0.09  -maxTreeDepthInLiterals -1  -rdnVersion gflRdn  2>&1 | tee /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train_output.txt


/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no classes that implement `__sklearn_tags__` in the MRO and/or all classes in the MRO call `super().__sklearn_tags__()`. Make sure to inherit from `BaseEstimator` which implements `__sklearn_tags__` (or alternatively define `__sklearn_tags__` but we don't recommend this approach). Note that `BaseEstimator` needs to be on the right side of other Mixins in the inheritance order. The default are now used instead since retrieving tags failed. This warning will be replaced by an error in 1.8.
  warnings.warn(
/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no cla

{'inference_time_ms': 800, 'f1': 0.585812, 'recall': 0.988417, 'precision': 0.41626, 'cll': -0.605624, 'auc_pr': 0.667603, 'auc_roc': 0.748204}
java  -jar /home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/srlearn/BoostSRL.jar -l -train /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train -target companyeconomicsector -trees 10 -negPosRatio 2 -focalLossAlpha 1  -focalLossGamma 4  -boxCoxLambda 0.09  -maxTreeDepthInLiterals -1  -rdnVersion gflRdn  2>&1 | tee /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train_output.txt


/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no classes that implement `__sklearn_tags__` in the MRO and/or all classes in the MRO call `super().__sklearn_tags__()`. Make sure to inherit from `BaseEstimator` which implements `__sklearn_tags__` (or alternatively define `__sklearn_tags__` but we don't recommend this approach). Note that `BaseEstimator` needs to be on the right side of other Mixins in the inheritance order. The default are now used instead since retrieving tags failed. This warning will be replaced by an error in 1.8.
  warnings.warn(
/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no cla

... (26 outputs left out)

{'inference_time_ms': 750, 'f1': 0.589651, 'recall': 0.945946, 'precision': 0.428322, 'cll': -0.615133, 'auc_pr': 0.674235, 'auc_roc': 0.758952}
java  -jar /home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/srlearn/BoostSRL.jar -l -train /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train -target companyeconomicsector -trees 10 -negPosRatio 2 -focalLossAlpha 1  -focalLossGamma 4  -boxCoxLambda 1.15  -maxTreeDepthInLiterals -1  -rdnVersion gflRdn  2>&1 | tee /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train_output.txt


/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no classes that implement `__sklearn_tags__` in the MRO and/or all classes in the MRO call `super().__sklearn_tags__()`. Make sure to inherit from `BaseEstimator` which implements `__sklearn_tags__` (or alternatively define `__sklearn_tags__` but we don't recommend this approach). Note that `BaseEstimator` needs to be on the right side of other Mixins in the inheritance order. The default are now used instead since retrieving tags failed. This warning will be replaced by an error in 1.8.
  warnings.warn(
/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no cla

{'inference_time_ms': 755, 'f1': 0.59118, 'recall': 0.957529, 'precision': 0.427586, 'cll': -0.615085, 'auc_pr': 0.670701, 'auc_roc': 0.753932}
java  -jar /home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/srlearn/BoostSRL.jar -l -train /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train -target companyeconomicsector -trees 10 -negPosRatio 2 -focalLossAlpha 1  -focalLossGamma 4  -boxCoxLambda 1.15  -maxTreeDepthInLiterals -1  -rdnVersion gflRdn  2>&1 | tee /home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning/bsrl_data/data0/train_output.txt


/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no classes that implement `__sklearn_tags__` in the MRO and/or all classes in the MRO call `super().__sklearn_tags__()`. Make sure to inherit from `BaseEstimator` which implements `__sklearn_tags__` (or alternatively define `__sklearn_tags__` but we don't recommend this approach). Note that `BaseEstimator` needs to be on the right side of other Mixins in the inheritance order. The default are now used instead since retrieving tags failed. This warning will be replaced by an error in 1.8.
  warnings.warn(
/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no cla

In [10]:
from models.main import default_workflow
from models.utils import update_experiment_state

base_path = "/home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning"

for target in targets:
    for config in configs:
        print(config)
        out = default_workflow(base_path, target, {**config, "run_original_rdn_boost": False}, base_path, small_dataset = False)
        update_experiment_state(os.path.join(base_path, f"large_dataset_results_gfl_{target}.json"), {**config, "run_original_rdn_boost": False, "results": out})
    

{'focal_loss_alpha': 0.5, 'focal_loss_gamma': 0, 'boxcox_lambda': 0}
java  -jar /home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/srlearn/BoostSRL.jar -l -train /home/alexandre/mestrado/graph-embedding/focal_loss_experiments/no_transfer_learning/bsrl_data/data0/train -target teamplayssport -trees 10 -negPosRatio 2 -focalLossAlpha 0.5  -focalLossGamma 0  -boxCoxLambda 0  -maxTreeDepthInLiterals 3  -runOriginalRdnBoost False  2>&1 | tee /home/alexandre/mestrado/graph-embedding/focal_loss_experiments/no_transfer_learning/bsrl_data/data0/train_output.txt


/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no classes that implement `__sklearn_tags__` in the MRO and/or all classes in the MRO call `super().__sklearn_tags__()`. Make sure to inherit from `BaseEstimator` which implements `__sklearn_tags__` (or alternatively define `__sklearn_tags__` but we don't recommend this approach). Note that `BaseEstimator` needs to be on the right side of other Mixins in the inheritance order. The default are now used instead since retrieving tags failed. This warning will be replaced by an error in 1.8.
  warnings.warn(
/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no cla

java  -jar /home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/srlearn/BoostSRL.jar -l -train /home/alexandre/mestrado/graph-embedding/focal_loss_experiments/no_transfer_learning/bsrl_data/data0/train -target teamplayssport -trees 10 -negPosRatio 2 -focalLossAlpha 0.5  -focalLossGamma 0  -boxCoxLambda 0  -maxTreeDepthInLiterals 3  -runOriginalRdnBoost False  2>&1 | tee /home/alexandre/mestrado/graph-embedding/focal_loss_experiments/no_transfer_learning/bsrl_data/data0/train_output.txt


/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no classes that implement `__sklearn_tags__` in the MRO and/or all classes in the MRO call `super().__sklearn_tags__()`. Make sure to inherit from `BaseEstimator` which implements `__sklearn_tags__` (or alternatively define `__sklearn_tags__` but we don't recommend this approach). Note that `BaseEstimator` needs to be on the right side of other Mixins in the inheritance order. The default are now used instead since retrieving tags failed. This warning will be replaced by an error in 1.8.
  warnings.warn(
/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no cla

java  -jar /home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/srlearn/BoostSRL.jar -l -train /home/alexandre/mestrado/graph-embedding/focal_loss_experiments/no_transfer_learning/bsrl_data/data0/train -target teamplayssport -trees 10 -negPosRatio 2 -focalLossAlpha 0.5  -focalLossGamma 0  -boxCoxLambda 0  -maxTreeDepthInLiterals 3  -runOriginalRdnBoost False  2>&1 | tee /home/alexandre/mestrado/graph-embedding/focal_loss_experiments/no_transfer_learning/bsrl_data/data0/train_output.txt


/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no classes that implement `__sklearn_tags__` in the MRO and/or all classes in the MRO call `super().__sklearn_tags__()`. Make sure to inherit from `BaseEstimator` which implements `__sklearn_tags__` (or alternatively define `__sklearn_tags__` but we don't recommend this approach). Note that `BaseEstimator` needs to be on the right side of other Mixins in the inheritance order. The default are now used instead since retrieving tags failed. This warning will be replaced by an error in 1.8.
  warnings.warn(
/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no cla

... (9,710 outputs left out)

java  -jar /home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/srlearn/BoostSRL.jar -l -train /home/alexandre/mestrado/graph-embedding/focal_loss_experiments/no_transfer_learning/bsrl_data/data0/train -target advisedby -trees 10 -negPosRatio 2 -focalLossAlpha 0.7  -focalLossGamma 3  -boxCoxLambda 1  -maxTreeDepthInLiterals 3  -runOriginalRdnBoost False  2>&1 | tee /home/alexandre/mestrado/graph-embedding/focal_loss_experiments/no_transfer_learning/bsrl_data/data0/train_output.txt


/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no classes that implement `__sklearn_tags__` in the MRO and/or all classes in the MRO call `super().__sklearn_tags__()`. Make sure to inherit from `BaseEstimator` which implements `__sklearn_tags__` (or alternatively define `__sklearn_tags__` but we don't recommend this approach). Note that `BaseEstimator` needs to be on the right side of other Mixins in the inheritance order. The default are now used instead since retrieving tags failed. This warning will be replaced by an error in 1.8.
  warnings.warn(
/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no cla

java  -jar /home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/srlearn/BoostSRL.jar -l -train /home/alexandre/mestrado/graph-embedding/focal_loss_experiments/no_transfer_learning/bsrl_data/data0/train -target advisedby -trees 10 -negPosRatio 2 -focalLossAlpha 0.7  -focalLossGamma 3  -boxCoxLambda 1  -maxTreeDepthInLiterals 3  -runOriginalRdnBoost False  2>&1 | tee /home/alexandre/mestrado/graph-embedding/focal_loss_experiments/no_transfer_learning/bsrl_data/data0/train_output.txt


/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no classes that implement `__sklearn_tags__` in the MRO and/or all classes in the MRO call `super().__sklearn_tags__()`. Make sure to inherit from `BaseEstimator` which implements `__sklearn_tags__` (or alternatively define `__sklearn_tags__` but we don't recommend this approach). Note that `BaseEstimator` needs to be on the right side of other Mixins in the inheritance order. The default are now used instead since retrieving tags failed. This warning will be replaced by an error in 1.8.
  warnings.warn(
/home/alexandre/mestrado/graph-embedding/venv_3.10.18/lib/python3.10/site-packages/sklearn/utils/_tags.py:338: DeprecationWarning: The following error was raised: 'RDNBoost' object has no attribute '__sklearn_tags__'. It seems that there are no cla

In [ ]:
from models.main import default_workflow
from models.utils import update_experiment_state

base_path = "/home/alexandre/mestrado/graph-embedding/focal_loss_experiments_2/no_transfer_learning"

for target in targets:
    if target != "twitter":
        continue
    out = default_workflow(base_path, target, {"rdn_version": "rdn"}, base_path, use_recursion=True)
    update_experiment_state(os.path.join(base_path, f"small_dataset_results_original_{target}_recursion.json"), {"rdn_version": "rdn", "results": out})    